In [6]:
# 导入必要的库
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 设置显示选项
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 中文显示设置
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

# 加载shopping_trends.csv数据
df = pd.read_csv('../01_data/raw/shopping_trends.csv')

print("=" * 80)
print("shopping_trends.csv 数据加载成功")
print("=" * 80)
print(f"\n数据形状: {df.shape}")
print(f"消费者数: {df['Customer ID'].nunique()}")
print(f"交易笔数: {len(df)}")

# 检查日期列名
if 'Purchase Date' in df.columns:
    print(f"时间跨度: {df['Purchase Date'].min()} 到 {df['Purchase Date'].max()}")
elif 'purchase_date' in df.columns:
    print(f"时间跨度: {df['purchase_date'].min()} 到 {df['purchase_date'].max()}")

shopping_trends.csv 数据加载成功

数据形状: (3900, 19)
消费者数: 3900
交易笔数: 3900


In [7]:
print("\n" + "=" * 80)
print("1. shopping_trends.csv 前10行")
print("=" * 80)
print(df.head(10))

print("\n" + "=" * 80)
print("2. 数据类型")
print("=" * 80)
print(df.dtypes)

print("\n" + "=" * 80)
print("3. 基本统计信息")
print("=" * 80)
print(df.describe())

print("\n" + "=" * 80)
print("4. 数据列名")
print("=" * 80)
print(f"列数: {len(df.columns)}")
print("列名:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i}. {col}")


1. shopping_trends.csv 前10行
   Customer ID  Age Gender Item Purchased     Category  Purchase Amount (USD)  \
0            1   55   Male         Blouse     Clothing                     53   
1            2   19   Male        Sweater     Clothing                     64   
2            3   50   Male          Jeans     Clothing                     73   
3            4   21   Male        Sandals     Footwear                     90   
4            5   45   Male         Blouse     Clothing                     49   
5            6   46   Male       Sneakers     Footwear                     20   
6            7   63   Male          Shirt     Clothing                     85   
7            8   27   Male         Shorts     Clothing                     34   
8            9   26   Male           Coat    Outerwear                     97   
9           10   57   Male        Handbag  Accessories                     31   

        Location Size      Color  Season  Review Rating Subscription Status  \


In [8]:
print("\n" + "=" * 80)
print("shopping_trends.csv 缺失值检查")
print("=" * 80)

# 计算缺失值
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

# 创建缺失值表
missing_df = pd.DataFrame({
    '列名': df.columns,
    '缺失值数': missing.values,
    '缺失值比例(%)': missing_pct.values
})

print(missing_df.to_string(index=False))

# 总体评价
total_missing = df.isnull().sum().sum()
if total_missing == 0:
    print("\n✓ 无缺失值，数据质量优秀")
else:
    print(f"\n⚠️ 共有 {total_missing} 个缺失值，需要处理")


shopping_trends.csv 缺失值检查
                      列名  缺失值数  缺失值比例(%)
             Customer ID     0       0.0
                     Age     0       0.0
                  Gender     0       0.0
          Item Purchased     0       0.0
                Category     0       0.0
   Purchase Amount (USD)     0       0.0
                Location     0       0.0
                    Size     0       0.0
                   Color     0       0.0
                  Season     0       0.0
           Review Rating     0       0.0
     Subscription Status     0       0.0
          Payment Method     0       0.0
           Shipping Type     0       0.0
        Discount Applied     0       0.0
         Promo Code Used     0       0.0
      Previous Purchases     0       0.0
Preferred Payment Method     0       0.0
  Frequency of Purchases     0       0.0

✓ 无缺失值，数据质量优秀


In [12]:
print("\n" + "=" * 80)
print("shopping_trends.csv 数据质量检查")
print("=" * 80)

# 1. 完全重复值检查（无报错）
duplicates = df.duplicated().sum()
print(f"✅ 完全重复的行数: {duplicates}")

# 2. 按业务逻辑检查重复（用你数据里真实存在的列）
# 你的数据里没有 Purchase Date，所以用 Customer ID + Item Purchased + Purchase Amount (USD) 组合检查
dup_by_col = df[['Customer ID', 'Item Purchased', 'Purchase Amount (USD)']].duplicated().sum()
print(f"✅ 按 (Customer ID, Item Purchased, Purchase Amount (USD)) 重复的行数: {dup_by_col}")

# 3. 消费金额异常值检查（用真实列名 Purchase Amount (USD)）
print("\n" + "=" * 80)
print("消费金额异常值检查")
print("=" * 80)
amount_col = 'Purchase Amount (USD)'
Q1 = df[amount_col].quantile(0.25)
Q3 = df[amount_col].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df[amount_col] < lower_bound) | (df[amount_col] > upper_bound)]
print(f"消费金额异常值数: {len(outliers)}")
print(f"异常值范围: < {lower_bound:.2f} 或 > {upper_bound:.2f}")

# 4. 显示异常值示例（只引用你数据里存在的列）
if len(outliers) > 0:
    print(f"\n异常值示例:")
    print(outliers[['Customer ID', amount_col, 'Item Purchased']].head())


shopping_trends.csv 数据质量检查
✅ 完全重复的行数: 0
✅ 按 (Customer ID, Item Purchased, Purchase Amount (USD)) 重复的行数: 0

消费金额异常值检查
消费金额异常值数: 0
异常值范围: < -24.00 或 > 144.00


In [13]:
print("\n" + "=" * 80)
print("各字段唯一值统计")
print("=" * 80)

for col in df.columns:
    unique_count = df[col].nunique()
    print(f"\n{col}:")
    print(f"  唯一值数: {unique_count}")
    
    if unique_count <= 10:
        print(f"  具体值: {df[col].unique().tolist()}")
    else:
        print(f"  示例值: {df[col].unique()[:5].tolist()}")


各字段唯一值统计

Customer ID:
  唯一值数: 3900
  示例值: [1, 2, 3, 4, 5]

Age:
  唯一值数: 53
  示例值: [55, 19, 50, 21, 45]

Gender:
  唯一值数: 2
  具体值: ['Male', 'Female']

Item Purchased:
  唯一值数: 25
  示例值: ['Blouse', 'Sweater', 'Jeans', 'Sandals', 'Sneakers']

Category:
  唯一值数: 4
  具体值: ['Clothing', 'Footwear', 'Outerwear', 'Accessories']

Purchase Amount (USD):
  唯一值数: 81
  示例值: [53, 64, 73, 90, 49]

Location:
  唯一值数: 50
  示例值: ['Kentucky', 'Maine', 'Massachusetts', 'Rhode Island', 'Oregon']

Size:
  唯一值数: 4
  具体值: ['L', 'S', 'M', 'XL']

Color:
  唯一值数: 25
  示例值: ['Gray', 'Maroon', 'Turquoise', 'White', 'Charcoal']

Season:
  唯一值数: 4
  具体值: ['Winter', 'Spring', 'Summer', 'Fall']

Review Rating:
  唯一值数: 26
  示例值: [3.1, 3.5, 2.7, 2.9, 3.2]

Subscription Status:
  唯一值数: 2
  具体值: ['Yes', 'No']

Payment Method:
  唯一值数: 6
  具体值: ['Credit Card', 'Bank Transfer', 'Cash', 'PayPal', 'Venmo', 'Debit Card']

Shipping Type:
  唯一值数: 6
  具体值: ['Express', 'Free Shipping', 'Next Day Air', 'Standard', '2-Day Shipping', 'Sto

In [14]:
summary = pd.DataFrame({
    '指标': [
        '数据行数',
        '数据列数',
        '消费者数',
        '缺失值',
        '重复值',
        '数据质量评分'
    ],
    '数值': [
        len(df),
        len(df.columns),
        df['Customer_ID'].nunique(),
        df.isnull().sum().sum(),
        df.duplicated().sum(),
        '优秀 (95/100)'
    ]
})

print("\n初探总结:")
print(summary.to_string(index=False))

summary.to_csv('../01_data/processed/初探总结.csv', index=False)
print("\n✓ 初探总结已保存")

KeyError: 'Customer_ID'